<a href="https://colab.research.google.com/github/kader-xai/llm-from-scratch/blob/main/notebooks/answer/Chapter05_layernorm_answer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🧱 **LLM From Scratch · Chapter 05 — LayerNorm — stabilise activations**

> ✅ Answer notebook — full working code.

## Goal

Deep networks train badly when activation magnitudes drift. **LayerNorm** normalises each token's
vector to zero mean / unit variance across its `C` features, then rescales with learnable `gamma`
(scale) and `beta` (shift). We implement it from scratch to demystify it.

In [ ]:
# ── Setup (run first) ─────────────────────────────────────────────────
# Pure PyTorch — no extra installs needed beyond torch (preinstalled on Colab).
import torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(1337)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", device)

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, C, eps=1e-5):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(C))
        self.beta  = nn.Parameter(torch.zeros(C))
        self.eps = eps
    def forward(self, x):                      # x: (B, T, C)
        mean = x.mean(-1, keepdim=True)
        var  = x.var(-1, keepdim=True, unbiased=False)
        xhat = (x - mean) / torch.sqrt(var + self.eps)
        return self.gamma * xhat + self.beta

ln = LayerNorm(32)
x = torch.randn(4, 8, 32) * 5 + 2              # off-centre, high variance
y = ln(x)
print("before: mean=%.2f std=%.2f" % (x.mean(), x.std()))
print("after:  mean=%.2f std=%.2f" % (y.mean(), y.std()))   # ~0 mean, ~1 std

✅ Activations stay tame. Now the heart of the transformer: attention.